# Sprint 4 - Transformer Models (NER + ABSA)

Fine-tune transformer untuk mengalahkan baseline Sprint 3 (NER CRF F1=0.8365, ABSA macro-F1=0.7669, gap multi-conflict=0.2057).

**Pilihan model (berbeda per task, by design):**
| Task | Model | Alasan |
|---|---|---|
| NER  | `bert-base-cased`   | Casing kritis (`BSE` != `bse`) untuk entity recognition |
| ABSA | `ProsusAI/finbert`  | Encoder pre-trained di financial corpus; head di-reinit ke 3 label kita |

**Lingkungan:** transformers 5.x (API: `eval_strategy`, `processing_class=`), torch 2.12 CPU.
Di mesin ber-GPU (RTX 4070 Ti), `Trainer` otomatis pakai CUDA - tidak perlu ubah kode.

**Catatan deadline:** kalau CPU terlalu lambat, turunkan `num_train_epochs` ke 2, atau pindahkan notebook ke mesin GPU.

In [1]:
import json, numpy as np, pandas as pd, torch
from pathlib import Path
from collections import Counter
import transformers
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          AutoModelForSequenceClassification, AutoConfig,
                          TrainingArguments, Trainer, set_seed,
                          DataCollatorForTokenClassification, DataCollatorWithPadding)
from datasets import load_dataset

set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers:', transformers.__version__, '| torch:', torch.__version__, '| device:', DEVICE)

PROJECT_ROOT   = Path('..').resolve()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR      = PROJECT_ROOT / 'backend' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MAX_LEN = 64  # headline max 23 kata -> aman (Sprint 1)

/home/anton/TextMining/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 5.10.1 | torch: 2.12.0+cu126 | device: cuda


## PART A - NER (`bert-base-cased`)

### A.1 Load data + label maps
BIO tags: `O`, `B-ENT`, `I-ENT`.

In [2]:
ner = load_dataset('json', data_files={
    'train': str(DATA_PROCESSED / 'ner_train.jsonl'),
    'val':   str(DATA_PROCESSED / 'ner_val.jsonl'),
    'test':  str(DATA_PROCESSED / 'ner_test.jsonl'),
})

LABELS = ['O', 'B-ENT', 'I-ENT']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
print(ner)
print('label2id:', label2id)

DatasetDict({
    train: Dataset({
        features: ['s_no', 'tokens', 'ner_tags'],
        num_rows: 8506
    })
    val: Dataset({
        features: ['s_no', 'tokens', 'ner_tags'],
        num_rows: 1064
    })
    test: Dataset({
        features: ['s_no', 'tokens', 'ner_tags'],
        num_rows: 1060
    })
})
label2id: {'O': 0, 'B-ENT': 1, 'I-ENT': 2}


### A.2 Tokenisasi + alignment subword (opsi B)

BERT memecah kata jadi subword (`Woolworth's` -> `Wool ##worth ' s`). BIO tag kita per-kata harus di-align:
- **Subword pertama** tiap kata -> dapat label aslinya.
- **Subword lanjutan** -> label `-100` (diabaikan `CrossEntropyLoss`, tidak ikut metrik).

`word_ids()` dari fast tokenizer memetakan tiap subword ke indeks kata aslinya.

In [3]:
tok_ner = AutoTokenizer.from_pretrained('bert-base-cased')

def tokenize_and_align(examples):
    tokenized = tok_ner(examples['tokens'], truncation=True, max_length=MAX_LEN,
                        is_split_into_words=True)
    all_labels = []
    for i, tags in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        prev, label_ids = None, []
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev:
                label_ids.append(label2id[tags[wid]])
            else:
                label_ids.append(-100)   # subword lanjutan
            prev = wid
        all_labels.append(label_ids)
    tokenized['labels'] = all_labels
    return tokenized

ner_tok = ner.map(tokenize_and_align, batched=True,
                  remove_columns=ner['train'].column_names)
print('Contoh alignment (headline 0):')
ex = ner_tok['train'][0]
toks = tok_ner.convert_ids_to_tokens(ex['input_ids'])
for t, l in list(zip(toks, ex['labels']))[:12]:
    print(f'  {t:14s} {id2label.get(l, "IGN(-100)") if l != -100 else "IGN(-100)"}')

Contoh alignment (headline 0):
  [CLS]          IGN(-100)
  M              B-ENT
  ##MT           IGN(-100)
  ##C            IGN(-100)
  Q              O
  ##2            IGN(-100)
  net            O
  loss           O
  at             O
  Rs             O
  10             O
  .              IGN(-100)


### A.3 Model, collator, metrik
`DataCollatorForTokenClassification` mem-padding dinamis input + label (`-100`) per batch. Metrik pakai `seqeval` (entity-level), konsisten dengan baseline CRF.

In [4]:
from seqeval.metrics import precision_score, recall_score, f1_score

model_ner = AutoModelForTokenClassification.from_pretrained(
    'bert-base-cased', num_labels=len(LABELS), id2label=id2label, label2id=label2id)
collator_ner = DataCollatorForTokenClassification(tok_ner)

def compute_metrics_ner(p):
    preds = np.argmax(p.predictions, axis=2)
    true_lab, true_pred = [], []
    for pred, lab in zip(preds, p.label_ids):
        cur_l = [id2label[l] for l in lab if l != -100]
        cur_p = [id2label[pr] for pr, l in zip(pred, lab) if l != -100]
        true_lab.append(cur_l); true_pred.append(cur_p)
    return {'precision': precision_score(true_lab, true_pred),
            'recall':    recall_score(true_lab, true_pred),
            'f1':        f1_score(true_lab, true_pred)}

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2045.89it/s]
[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not 

In [6]:
args_ner = TrainingArguments(
    output_dir=str(MODEL_DIR / 'ner_bert_ckpt'),
    eval_strategy='epoch', save_strategy='epoch',
    learning_rate=2e-5, num_train_epochs=5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    weight_decay=0.01, logging_steps=50,
    load_best_model_at_end=True, metric_for_best_model='f1', greater_is_better=True,
    save_total_limit=1, report_to='none', seed=42,
)
trainer_ner = Trainer(
    model=model_ner, args=args_ner,
    train_dataset=ner_tok['train'], eval_dataset=ner_tok['val'],
    data_collator=collator_ner, processing_class=tok_ner,
    compute_metrics=compute_metrics_ner,
)
trainer_ner.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.087209,0.082950,0.879863,0.904561,0.892042
2,0.054166,0.089546,0.888353,0.904561,0.896384
3,0.032800,0.093755,0.891799,0.908070,0.899861
4,0.022543,0.109953,0.896312,0.903860,0.900070
5,0.014655,0.120009,0.894077,0.900351,0.897203


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.10it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

TrainOutput(global_step=2660, training_loss=0.052500219766358686, metrics={'train_runtime': 187.4287, 'train_samples_per_second': 226.913, 'train_steps_per_second': 14.192, 'total_flos': 574487548043508.0, 'train_loss': 0.052500219766358686, 'epoch': 5.0})

In [7]:
# Evaluasi pada TEST + report entity-level
pred = trainer_ner.predict(ner_tok['test'])
print('NER BERT - TEST metrics:', {k: round(v, 4) for k, v in pred.metrics.items() if 'f1' in k or 'prec' in k or 'rec' in k})

from seqeval.metrics import classification_report as seq_report
y = np.argmax(pred.predictions, axis=2)
tl = [[id2label[l] for l in lab if l != -100] for lab in pred.label_ids]
tp = [[id2label[p] for p, l in zip(pr, lab) if l != -100] for pr, lab in zip(y, pred.label_ids)]
print(seq_report(tl, tp))
print('Baseline CRF F1 = 0.8365  ->  BERT F1 =', round(f1_score(tl, tp), 4))

NER BERT - TEST metrics: {'test_precision': 0.9067, 'test_recall': 0.9231, 'test_f1': 0.9148}
              precision    recall  f1-score   support

         ENT       0.91      0.92      0.91      1431

   micro avg       0.91      0.92      0.91      1431
   macro avg       0.91      0.92      0.91      1431
weighted avg       0.91      0.92      0.91      1431

Baseline CRF F1 = 0.8365  ->  BERT F1 = 0.9148


In [8]:
trainer_ner.save_model(str(MODEL_DIR / 'ner_bert_final'))
tok_ner.save_pretrained(str(MODEL_DIR / 'ner_bert_final'))
print('Saved NER model ->', MODEL_DIR / 'ner_bert_final')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.68it/s]


Saved NER model -> /home/anton/TextMining/Financial-News-Miner/backend/models/ner_bert_final


## PART B - ABSA (`ProsusAI/finbert`)

### B.1 Format input: `[CLS] title [SEP] entity [SEP]`

Inilah keputusan arsitektural inti. Saat memanggil `tokenizer(title, entity)`, BERT **otomatis** membentuk `[CLS] title [SEP] entity [SEP]` dengan `token_type_ids` yang benar (0 untuk title, 1 untuk entity). Self-attention bisa fokus ke konteks yang relevan untuk entity target -> inilah yang menutup gap multi-conflict pada baseline BoW.

`ProsusAI/finbert` punya head 3-kelas bawaan dengan urutan berbeda; kita pakai `ignore_mismatched_sizes=True` agar head di-reinit sesuai label kita (`0=negative, 1=neutral, 2=positive`).

In [9]:
absa = load_dataset('csv', data_files={
    'train': str(DATA_PROCESSED / 'absa_train.csv'),
    'val':   str(DATA_PROCESSED / 'absa_val.csv'),
    'test':  str(DATA_PROCESSED / 'absa_test.csv'),
})
ABSA_ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}
ABSA_LABEL2ID = {v: k for k, v in ABSA_ID2LABEL.items()}

tok_absa = AutoTokenizer.from_pretrained('ProsusAI/finbert')

def tokenize_pair(ex):
    enc = tok_absa(ex['title'], ex['entity'], truncation=True, max_length=MAX_LEN)
    enc['labels'] = ex['label']
    return enc

absa_tok = absa.map(tokenize_pair, batched=True,
                    remove_columns=absa['train'].column_names)
# Cek pembentukan [CLS] title [SEP] entity [SEP]
ex = absa_tok['train'][0]
print('Decoded:', tok_absa.decode(ex['input_ids']))
print('token_type_ids:', ex['token_type_ids'][:20])

Generating train split: 11433 examples [00:00, 475577.97 examples/s]
Generating val split: 1430 examples [00:00, 357525.91 examples/s]
Generating test split: 1448 examples [00:00, 420359.37 examples/s]
Map: 100%|██████████| 1448/1448 [00:00<00:00, 72094.11 examples/s]

Decoded: [CLS] mmtc q2 net loss at rs 10. 4 crore [SEP] mmtc [SEP]
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1]


### B.2 Class weights + Trainer berbobot

Imbalance (negative paling sedikit) ditangani dengan `CrossEntropyLoss(weight=...)`, bobot = `n_total / (n_kelas * count_kelas)` - rumus 'balanced' yang sama dengan sklearn. Kita override `compute_loss` di subclass `Trainer`.

In [10]:
import torch.nn as nn

counts = Counter(absa['train']['label'])
N, K = sum(counts.values()), 3
class_weights = torch.tensor([N / (K * counts[i]) for i in range(K)], dtype=torch.float)
print('Class counts:', dict(counts))
print('Class weights:', [round(w, 3) for w in class_weights.tolist()])

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits.view(-1, K), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

Class counts: {1: 4341, 2: 4048, 0: 3044}
Class weights: [1.252, 0.878, 0.941]


In [11]:
from sklearn.metrics import f1_score as skf1, accuracy_score

model_absa = AutoModelForSequenceClassification.from_pretrained(
    'ProsusAI/finbert', num_labels=K,
    id2label=ABSA_ID2LABEL, label2id=ABSA_LABEL2ID,
    ignore_mismatched_sizes=True)
collator_absa = DataCollatorWithPadding(tok_absa)

def compute_metrics_absa(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': accuracy_score(p.label_ids, preds),
            'macro_f1': skf1(p.label_ids, preds, average='macro')}

args_absa = TrainingArguments(
    output_dir=str(MODEL_DIR / 'absa_finbert_ckpt'),
    eval_strategy='epoch', save_strategy='epoch',
    learning_rate=2e-5, num_train_epochs=5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    weight_decay=0.01, logging_steps=50,
    load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
    save_total_limit=1, report_to='none', seed=42,
)
trainer_absa = WeightedTrainer(
    class_weights=class_weights,
    model=model_absa, args=args_absa,
    train_dataset=absa_tok['train'], eval_dataset=absa_tok['val'],
    data_collator=collator_absa, processing_class=tok_absa,
    compute_metrics=compute_metrics_absa,
)
trainer_absa.train()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15160.95it/s]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.382834,0.391478,0.864336,0.865527
2,0.275805,0.420363,0.871329,0.871702
3,0.205523,0.567355,0.873427,0.873787
4,0.104134,0.661167,0.867832,0.868840
5,0.053597,0.701312,0.869231,0.870440


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.25it/s]


TrainOutput(global_step=3575, training_loss=0.20750923408494962, metrics={'train_runtime': 250.4078, 'train_samples_per_second': 228.288, 'train_steps_per_second': 14.277, 'total_flos': 896111369761986.0, 'train_loss': 0.20750923408494962, 'epoch': 5.0})

In [12]:
from sklearn.metrics import classification_report, confusion_matrix

pred_absa = trainer_absa.predict(absa_tok['test'])
y_pred = np.argmax(pred_absa.predictions, axis=1)
y_true = pred_absa.label_ids
names = ['negative', 'neutral', 'positive']
print('=== ABSA FinBERT - TEST ===')
print(classification_report(y_true, y_pred, target_names=names, digits=4))
cm = confusion_matrix(y_true, y_pred)
print(pd.DataFrame(cm, index=[f'true_{n}' for n in names], columns=[f'pred_{n}' for n in names]))
print('\nBaseline SVM macro-F1 = 0.7669  ->  FinBERT macro-F1 =',
      round(skf1(y_true, y_pred, average='macro'), 4))

=== ABSA FinBERT - TEST ===
              precision    recall  f1-score   support

    negative     0.8903    0.8880    0.8892       384
     neutral     0.8302    0.8923    0.8602       548
    positive     0.9223    0.8508    0.8851       516

    accuracy                         0.8764      1448
   macro avg     0.8809    0.8770    0.8781      1448
weighted avg     0.8790    0.8764    0.8767      1448

               pred_negative  pred_neutral  pred_positive
true_negative            341            35              8
true_neutral              30           489             29
true_positive             12            65            439

Baseline SVM macro-F1 = 0.7669  ->  FinBERT macro-F1 = 0.8781


### B.3 Breakdown single vs multi-conflict (apple-to-apple dgn Sprint 3)

Pertanyaan kunci: apakah transformer menutup gap 0.2057 yang dipunyai BoW?

In [13]:
te = pd.read_csv(DATA_PROCESSED / 'absa_test.csv').copy()
te['pred'] = y_pred
te['correct'] = te['pred'] == te['label']
g = te.groupby('s_no')
ne = g['entity'].transform('size'); ns = g['sentiment'].transform('nunique')
te['head_type'] = ['single' if e == 1 else ('multi-conflict' if s > 1 else 'multi-uniform')
                   for e, s in zip(ne, ns)]
summary = te.groupby('head_type').agg(n_pairs=('correct','size'), accuracy=('correct','mean')).round(4)
print(summary)
gap = round(summary.loc['single','accuracy'] - summary.loc['multi-conflict','accuracy'], 4)
print(f'\nGap single vs multi-conflict: FinBERT={gap}  (baseline BoW=0.2057)')

                n_pairs  accuracy
head_type                        
multi-conflict      294    0.8571
multi-uniform       376    0.8590
single              778    0.8920

Gap single vs multi-conflict: FinBERT=0.0349  (baseline BoW=0.2057)


In [14]:
trainer_absa.save_model(str(MODEL_DIR / 'absa_finbert_final'))
tok_absa.save_pretrained(str(MODEL_DIR / 'absa_finbert_final'))
print('Saved ABSA model ->', MODEL_DIR / 'absa_finbert_final')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.03it/s]


Saved ABSA model -> /home/anton/TextMining/Financial-News-Miner/backend/models/absa_finbert_final


## Ringkasan Sprint 4 (isi setelah run)

| Task | Baseline (Sprint 3) | Transformer (Sprint 4) | Delta |
|---|---|---|---|
| NER (entity-F1)        | 0.8365 (CRF)     | ___ (bert-cased)  | ___ |
| ABSA (macro-F1)        | 0.7669 (SVM)     | ___ (finbert)     | ___ |
| ABSA gap multi-conflict| 0.2057           | ___               | ___ |

**Interpretasi:** jika gap multi-conflict mengecil signifikan, itu bukti `[CLS] title [SEP] entity [SEP]` berhasil membuat model aspect-aware - kontribusi ilmiah utama proyek.

**Next (Sprint 5):** pipeline end-to-end (headline -> NER -> per-entity ABSA -> {entity: sentiment}).